### Importación

In [1]:
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import time
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize, LabelEncoder, StandardScaler
from sklearn.metrics import roc_curve, auc, ConfusionMatrixDisplay
from sklearn.multiclass import OneVsRestClassifier
import matplotlib.pyplot as plt

### Carga dataset

In [ ]:
def cargar_datos(
    carpeta="images_training_rev1",
    archivo_clasificaciones="training_solutions_rev1.csv",
    max_imagenes=60000,
    size=(128, 128)
):
    # 1. Leer archivo CSV con clasificaciones
    df = pd.read_csv(archivo_clasificaciones)
    print(f"Archivo CSV cargado con {len(df)} filas.")

    # Verificar columna identificadora
    if "GalaxyID" not in df.columns:
        raise ValueError("El archivo CSV debe tener una columna 'GalaxyID' que identifique las imágenes.")

    # Crear lista de IDs disponibles
    galaxy_ids = df["GalaxyID"].astype(str).tolist()

    # 2. Listar imágenes disponibles
    archivos = [f for f in os.listdir(carpeta) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

    # Filtrar solo imágenes cuyo GalaxyID exista en el CSV
    archivos_validos = [f for f in archivos if f.split(".")[0] in galaxy_ids]

    # Tomar muestra aleatoria
    if len(archivos_validos) > max_imagenes:
        archivos_validos = random.sample(archivos_validos, max_imagenes)

    print(f"Se seleccionaron {len(archivos_validos)} imágenes aleatorias.")

    # 3. Cargar imágenes y asociarlas con su etiqueta
    X, y = [], []

    for nombre in archivos_validos:
        galaxy_id = nombre.split(".")[0]
        fila = df[df["GalaxyID"] == int(galaxy_id)].iloc[0]
        etiqueta = fila.drop("GalaxyID").idxmax()  # clasificación más probable

        ruta = os.path.join(carpeta, nombre)
        try:
            img = Image.open(ruta).convert("L").resize(size)
            X.append(np.array(img))
            y.append(etiqueta)
        except Exception as e:
            print(f"Error al cargar {nombre}: {e}")

    X = np.array(X)
    print(f"Se cargaron {len(X)} imágenes etiquetadas de '{carpeta}'.")
    return X, y


# Ejemplo de uso
if __name__ == "__main__":
    X, y = cargar_datos(
        carpeta="images_training_rev1",
        archivo_clasificaciones="training_solutions_rev1.csv",
        max_imagenes=12000,
        size=(128, 128)
    )

# 1. Comparación según tipo de color 

## Preparación de datos para Tabla 1: Comparación RGB vs. Escala de grises

A continuación se muestra la preparación base de los datos para ambos casos:
- **RGB:** Cada imagen se aplana a un vector de 3 canales (128x128x3).
- **Gris:** Cada imagen se convierte a escala de grises y se aplana (128x128).

En ambos casos:
- Se codifican las etiquetas.
- Se divide el dataset en entrenamiento y prueba (70-30, mismo random_state).
- Se escalan los datos (StandardScaler).

Primero, preparación para imágenes RGB:

In [ ]:
# --- Preparación para imágenes RGB ---
# Cargar imágenes en RGB y etiquetas
def cargar_datos_rgb(carpeta, archivo_clasificaciones, max_imagenes=12000, size=(128,128)):
    df = pd.read_csv(archivo_clasificaciones)
    galaxy_ids = df["GalaxyID"].astype(str).tolist()
    archivos = [f for f in os.listdir(carpeta) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
    archivos_validos = [f for f in archivos if f.split(".")[0] in galaxy_ids]
    if len(archivos_validos) > max_imagenes:
        archivos_validos = random.sample(archivos_validos, max_imagenes)
    X, y = [], []
    for nombre in archivos_validos:
        galaxy_id = nombre.split(".")[0]
        fila = df[df["GalaxyID"] == int(galaxy_id)].iloc[0]
        etiqueta = fila.drop("GalaxyID").idxmax()
        ruta = os.path.join(carpeta, nombre)
        try:
            img = Image.open(ruta).convert("RGB").resize(size)
            X.append(np.array(img))
            y.append(etiqueta)
        except Exception as e:
            print(f"Error al cargar {nombre}: {e}")
    X = np.array(X)
    return X, y

# Cargar datos RGB
dir_imgs = "images_training_rev1"
file_labels = "training_solutions_rev1.csv"
X_rgb, y_rgb = cargar_datos_rgb(dir_imgs, file_labels)

# Aplanar imágenes RGB (N, 128, 128, 3) -> (N, 49152)
X_rgb_flat = X_rgb.reshape(X_rgb.shape[0], -1)

# Codificar etiquetas
le = LabelEncoder()
y_rgb_enc = le.fit_transform(y_rgb)

# División 70-30
Xtr_rgb, Xte_rgb, ytr_rgb, yte_rgb = train_test_split(X_rgb_flat, y_rgb_enc, test_size=0.3, random_state=42, stratify=y_rgb_enc)

# Escalado
scaler_rgb = StandardScaler()
Xtr_rgb_scaled = scaler_rgb.fit_transform(Xtr_rgb)
Xte_rgb_scaled = scaler_rgb.transform(Xte_rgb)

print(f"RGB: Xtr={Xtr_rgb_scaled.shape}, Xte={Xte_rgb_scaled.shape}, ytr={ytr_rgb.shape}, yte={yte_rgb.shape}")

---

Ahora, preparación para imágenes en escala de grises:

In [3]:
# --- Preparación para imágenes en escala de grises ---
def cargar_datos_gray(carpeta, archivo_clasificaciones, max_imagenes=12000, size=(128,128)):
    df = pd.read_csv(archivo_clasificaciones)
    galaxy_ids = df["GalaxyID"].astype(str).tolist()
    archivos = [f for f in os.listdir(carpeta) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
    archivos_validos = [f for f in archivos if f.split(".")[0] in galaxy_ids]
    if len(archivos_validos) > max_imagenes:
        archivos_validos = random.sample(archivos_validos, max_imagenes)
    X, y = [], []
    for nombre in archivos_validos:
        galaxy_id = nombre.split(".")[0]
        fila = df[df["GalaxyID"] == int(galaxy_id)].iloc[0]
        etiqueta = fila.drop("GalaxyID").idxmax()
        ruta = os.path.join(carpeta, nombre)
        try:
            img = Image.open(ruta).convert("L").resize(size)
            X.append(np.array(img))
            y.append(etiqueta)
        except Exception as e:
            print(f"Error al cargar {nombre}: {e}")
    X = np.array(X)
    return X, y

# Cargar datos en escala de grises
X_gray, y_gray = cargar_datos_gray(dir_imgs, file_labels)

# Aplanar imágenes gris (N, 128, 128) -> (N, 16384)
X_gray_flat = X_gray.reshape(X_gray.shape[0], -1)

# Codificar etiquetas
le_gray = LabelEncoder()
y_gray_enc = le_gray.fit_transform(y_gray)

# División 70-30
Xtr_gray, Xte_gray, ytr_gray, yte_gray = train_test_split(X_gray_flat, y_gray_enc, test_size=0.3, random_state=42, stratify=y_gray_enc)

# Escalado
scaler_gray = StandardScaler()
Xtr_gray_scaled = scaler_gray.fit_transform(Xtr_gray)
Xte_gray_scaled = scaler_gray.transform(Xte_gray)

print(f"Gray: Xtr={Xtr_gray_scaled.shape}, Xte={Xte_gray_scaled.shape}, ytr={ytr_gray.shape}, yte={yte_gray.shape}")

NameError: name 'dir_imgs' is not defined